In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv('fashion-mnist_train.csv')
df.head(3)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,9,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,6,0,0,0,0,0,0,0,5,0,...,0.0,0.0,0.0,30.0,43.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
from sklearn.model_selection import train_test_split
X = df.drop('label', axis=1)
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
X_train = X_train/255
X_test = X_test/255

In [ ]:
X_train = X_train.to_numpy()
X_test = X_test.to_numpy()
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

In [ ]:
import torch
X_train = torch.from_numpy(X_train).to(torch.float32)
X_test = torch.from_numpy(X_test).to(torch.float32)
y_train = torch.from_numpy(y_train).to(torch.long)
y_test = torch.from_numpy(y_test).to(torch.long)

In [ ]:
from torch.utils.data import Dataset, DataLoader
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [ ]:
train_ds = CustomDataset(X_train, y_train)
test_ds = CustomDataset(X_test, y_test)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, pin_memory=True)

In [ ]:
import torch
import torch.nn as nn

class Neural_Network(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(p=0.5),
        nn.Linear(128, 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(64, 10)
    )

  def forward(self, num_features):
    return self.model(num_features)

In [ ]:
learning_rate = 0.1
epochs = 100

In [ ]:
model = Neural_Network(X_train.shape[1])
model = model.to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

for epoch in range(epochs):
  for batch_features, batch_labels in train_loader:
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)
    y_pred = model(batch_features)
    loss = loss_function(y_pred, batch_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  print(f'Epoch: {epoch+1}, Loss:{loss.item()}')

Epoch: 1, Loss:0.49003568291664124
Epoch: 2, Loss:0.6517163515090942
Epoch: 3, Loss:0.7020817995071411
Epoch: 4, Loss:0.5482025742530823
Epoch: 5, Loss:0.6248109936714172
Epoch: 6, Loss:0.4581722021102905
Epoch: 7, Loss:0.753822386264801
Epoch: 8, Loss:0.6112238764762878
Epoch: 9, Loss:0.3835717439651489
Epoch: 10, Loss:0.6573600769042969
Epoch: 11, Loss:0.4310493767261505
Epoch: 12, Loss:0.45381495356559753
Epoch: 13, Loss:0.722718358039856
Epoch: 14, Loss:0.4505324363708496
Epoch: 15, Loss:0.20433856546878815
Epoch: 16, Loss:0.356374055147171
Epoch: 17, Loss:0.2604963183403015
Epoch: 18, Loss:0.3563956022262573
Epoch: 19, Loss:0.319024920463562
Epoch: 20, Loss:0.47336429357528687
Epoch: 21, Loss:0.45953109860420227
Epoch: 22, Loss:0.1960584819316864
Epoch: 23, Loss:0.17681708931922913
Epoch: 24, Loss:0.3501178026199341
Epoch: 25, Loss:0.3681419789791107
Epoch: 26, Loss:0.453617662191391
Epoch: 27, Loss:0.47505995631217957
Epoch: 28, Loss:0.3947541117668152
Epoch: 29, Loss:0.459030449

In [23]:
# Train accuracy
model.eval()
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)
    y_pred = model(batch_features)
    _, predicted = torch.max(y_pred, 1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()
print(f'Accuracy: {correct/total}')

Accuracy: 0.8633333333333333
